In [14]:
import re
import pandas as pd 
from typing import List, Dict, Tuple
from collections import defaultdict 
from nltk.tokenize import word_tokenize, sent_tokenize


In [15]:
train_text = """ I love Deep Learning. 
And I love NLP, with or without deep learning. 
In general I love AI, but I don't like A.B.C."""

# train_text = train_text.lower()

pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
train_tokens = re.findall(pattern, train_text)
train_tokens

['I',
 'love',
 'Deep',
 'Learning',
 '.',
 'And',
 'I',
 'love',
 'NLP',
 'with',
 'or',
 'without',
 'deep',
 'learning',
 '.',
 'In',
 'general',
 'I',
 'love',
 'AI',
 'but',
 'I',
 "don't",
 'like',
 'A.B.C.']

In [16]:

def add_boundaries(tokens: List[str], n: int) -> List[str]:
    """ 
    Add <S> and <E> Tokens for given n_gram size.
    """
    tokens_ = []
    start_tokens = ['<S>'] * (n) 
    end_tokens = ['<E>'] * (n)
    tokens_.extend(start_tokens)

    for i, token in enumerate(tokens):
        tokens_.append(token)
        if token == ".":
            tokens_.extend(end_tokens)
            # add start tokens if it is not the last token
            if i < len(tokens) - 1:
                tokens_.extend(start_tokens)

    # if the last token is not a period, add end tokens
    if tokens and tokens[-1] != ".":
        tokens_.extend(end_tokens)
    return tokens_



In [17]:
train_tokens_ = add_boundaries(train_tokens, n=1)

train_tokens_

['<S>',
 'I',
 'love',
 'Deep',
 'Learning',
 '.',
 '<E>',
 '<S>',
 'And',
 'I',
 'love',
 'NLP',
 'with',
 'or',
 'without',
 'deep',
 'learning',
 '.',
 '<E>',
 '<S>',
 'In',
 'general',
 'I',
 'love',
 'AI',
 'but',
 'I',
 "don't",
 'like',
 'A.B.C.',
 '<E>']

In [18]:

def ngram_counts(tokens: List[str], n: int) -> Dict[Tuple[str, ...], Dict[str, int]]:
    """ 
    Compute the ngram counts of given tokens with specifed context size.
    
    Args:
        tokens: list of tokenized strings.
        n size of context, n=1 >> bigram, n=2 >> trigram etc. 

    Returns:
        Nested Dict of (context, next_word) counts.
    """
    counts = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - n):
        context = tuple(tokens[i: i + n])
        next_ = tokens[i+n]
        counts[context][next_] += 1
    return counts
    

In [19]:
unicnt = ngram_counts(train_tokens, n=0)
bicnt = ngram_counts(train_tokens, n=1)
tricnt = ngram_counts(train_tokens, n=2)


In [20]:
unicnt


defaultdict(<function __main__.ngram_counts.<locals>.<lambda>()>,
            {(): defaultdict(int,
                         {'I': 4,
                          'love': 3,
                          'Deep': 1,
                          'Learning': 1,
                          '.': 2,
                          'And': 1,
                          'NLP': 1,
                          'with': 1,
                          'or': 1,
                          'without': 1,
                          'deep': 1,
                          'learning': 1,
                          'In': 1,
                          'general': 1,
                          'AI': 1,
                          'but': 1,
                          "don't": 1,
                          'like': 1,
                          'A.B.C.': 1})})

In [21]:
bicnt

defaultdict(<function __main__.ngram_counts.<locals>.<lambda>()>,
            {('I',): defaultdict(int, {'love': 3, "don't": 1}),
             ('love',): defaultdict(int, {'Deep': 1, 'NLP': 1, 'AI': 1}),
             ('Deep',): defaultdict(int, {'Learning': 1}),
             ('Learning',): defaultdict(int, {'.': 1}),
             ('.',): defaultdict(int, {'And': 1, 'In': 1}),
             ('And',): defaultdict(int, {'I': 1}),
             ('NLP',): defaultdict(int, {'with': 1}),
             ('with',): defaultdict(int, {'or': 1}),
             ('or',): defaultdict(int, {'without': 1}),
             ('without',): defaultdict(int, {'deep': 1}),
             ('deep',): defaultdict(int, {'learning': 1}),
             ('learning',): defaultdict(int, {'.': 1}),
             ('In',): defaultdict(int, {'general': 1}),
             ('general',): defaultdict(int, {'I': 1}),
             ('AI',): defaultdict(int, {'but': 1}),
             ('but',): defaultdict(int, {'I': 1}),
             ("don't

In [22]:
unicnt_df = pd.DataFrame.from_dict(unicnt, orient="index").fillna(0)
unicnt_df

,I,love,Deep,Learning,.,And,NLP,with,or,without,deep,learning,In,general,AI,but,don't,like,A.B.C.
(),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
bicnt_df = pd.DataFrame.from_dict(bicnt, orient="index").fillna(0)
bicnt_df

,love,don't,Deep,NLP,AI,Learning,.,And,In,I,with,or,without,deep,learning,general,but,like,A.B.C.
I,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
love,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Deep,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Learning,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
learning,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
.,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
And,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
general,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
but,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NLP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
bicnt_prob = bicnt_df.div(bicnt_df.sum(axis=1), axis=0)
bicnt_prob

,love,don't,Deep,NLP,AI,Learning,.,And,In,I,with,or,without,deep,learning,general,but,like,A.B.C.
I,0.75,0.25,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
love,0.00,0.00,0.333333,0.333333,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Deep,0.00,0.00,0.000000,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Learning,0.00,0.00,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
learning,0.00,0.00,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
.,0.00,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
And,0.00,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
general,0.00,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
but,0.00,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NLP,0.00,0.00,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Perplixity:

To compute the perplexity you should first tokenize the test text, take care of the beginings and endings of sentences, obtain the probalities of test tokens from the table above and insert them to the formula. 